# Biohub LB893 post-processing ablation controller

This lightweight notebook records the validated LB893 baseline output, defines the component ablation matrix, and emits a reproducible plan for Kaggle validation runs. Use `biohub-lb893-validation-ablation.ipynb` for the actual GPU/metric run.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
LB893_STATS = ROOT / "references/lb893-v1-output/run_stats.csv"
PRIOR_LEARNED_STATS = ROOT / "references/learned-candidate-v1-output/learned_test_stats.csv"
print("Workspace:", ROOT)
print("LB893 stats exists:", LB893_STATS.exists())


In [ ]:
lb893 = pd.read_csv(LB893_STATS)
summary = {
    "rows": int(lb893["nodes"].sum() + lb893["edges"].sum()),
    "nodes": int(lb893["nodes"].sum()),
    "edges": int(lb893["edges"].sum()),
    "division_like_sources": int(lb893["division_like_sources"].sum()),
    "motion_relink_edges": int(lb893["motion_relink_edges"].sum()),
    "gap_added_nodes": int(lb893["gap_added_nodes"].sum()),
    "gap2_added_nodes": int(lb893["gap2_added_nodes"].sum()),
    "safe_divisions_added": int(lb893["safe_divisions_added"].sum()),
    "linefit_smoothed_nodes": int(lb893["linefit_smoothed_nodes"].sum()),
}
print(json.dumps(summary, indent=2))
display(lb893[["dataset", "raw_nodes", "nodes", "raw_edges", "edges", "division_like_sources", "gap_added_nodes", "gap2_added_nodes", "safe_divisions_added", "linefit_smoothed_nodes"]])


In [ ]:
ABLATIONS = [
    {
        "name": "full_lb893",
        "rationale": "Reproduce public LB 0.893 baseline on exact train validation.",
        "env": {},
    },
    {
        "name": "no_safe_divisions",
        "rationale": "Measure whether the 381 added division edges are net positive under the exact division metric.",
        "env": {"BIOHUB_OUTPUT_SAFE_DIVISIONS": "0"},
    },
    {
        "name": "no_gap2",
        "rationale": "Re-test gap-2 only in the learned/motion-relinked context, where it may work despite failing classically.",
        "env": {"BIOHUB_OUTPUT_GAP2_RECOVERY": "0"},
    },
    {
        "name": "no_gap_close",
        "rationale": "Quantify the one-frame synthetic/reused gap nodes and their edge contribution.",
        "env": {"BIOHUB_OUTPUT_GAP_CLOSE": "0"},
    },
    {
        "name": "no_linefit",
        "rationale": "Check whether smoothing improves 7 um endpoint matching or only changes geometry cosmetically.",
        "env": {"BIOHUB_OUTPUT_LINEFIT_SMOOTH": "0"},
    },
    {
        "name": "no_motion_relink",
        "rationale": "Most important ablation: compare raw learned/ILP edges against the motion-relinked graph.",
        "env": {"BIOHUB_OUTPUT_MOTION_RELINK": "0"},
    },
]
plan = pd.DataFrame([{"name": a["name"], "rationale": a["rationale"], "env": json.dumps(a["env"], sort_keys=True)} for a in ABLATIONS])
display(plan)
(ROOT / "references/lb893-v1-output/ablation_plan.json").write_text(json.dumps(ABLATIONS, indent=2))
print("Wrote", ROOT / "references/lb893-v1-output/ablation_plan.json")


## Execution protocol

For each row in `ABLATIONS`, run `biohub-lb893-validation-ablation.ipynb` on Kaggle with T4 GPU, no internet, the competition data source, and `pilkwang/biohub-tracking-support-pack-50ep-v1`. Keep `BIOHUB_VALIDATION_MODE=1` and `BIOHUB_VALIDATION_MAX_EMBRYOS=2` for fast exact validation. Apply the row-specific environment overrides before running.

Only after the winning ablation beats `full_lb893` on exact validation should we create a test-submission notebook by setting `BIOHUB_VALIDATION_MODE=0` and using the test directory.